# Native RAG-Based Clinical Hallucination Detection Model
This notebook uses **ipywidgets** for a seamless, completely native UI inside Google Colab.

**Instructions:**
1. Ensure you are connected to a **T4 GPU** runtime (Runtime -> Change runtime type -> T4 GPU).
2. Run all the cells sequentially.
3. Use the Interactive GUI at the bottom to upload text or generate random abstracts!


In [ ]:
!pip install -q transformers accelerate bitsandbytes
!pip install -q langchain langchain-community langchain-text-splitters faiss-cpu sentence-transformers
!pip install -q datasets ipywidgets

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# #Optional: If you don't use Colab Secrets, uncomment below to login
# from huggingface_hub import login
# login(token="HF_TOKEN")


In [ ]:
import os
import random
from typing import List, Dict
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import CrossEncoder
from datasets import load_dataset
import nltk
nltk.download('punkt', quiet=True)
from nltk.tokenize import sent_tokenize

# Global Models
embeddings = None
llm_model = None
llm_tokenizer = None
nli_model = None

def initialize_models():
    global embeddings, llm_model, llm_tokenizer, nli_model
    if embeddings is not None:
        return # Already loaded

    print("Loading Embeddings...")
    embeddings = HuggingFaceEmbeddings(model_name="NeuML/pubmedbert-base-embeddings")

    print("Loading NLI Verifier...")
    nli_model = CrossEncoder('cross-encoder/nli-deberta-v3-base', max_length=512, device='cuda' if torch.cuda.is_available() else 'cpu')

    print("Loading Llama-3.2-3B-Instruct...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )

    # Needs HF_TOKEN in environment or logged in
    model_id = "meta-llama/Llama-3.2-3B-Instruct"
    llm_tokenizer = AutoTokenizer.from_pretrained(model_id)
    llm_model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto"
    )
    print("Models loaded successfully!")

# Initialize the models right away
initialize_models()

In [ ]:
def get_random_pubmed_abstract() -> str:
    ds = load_dataset('uiyunkim-hub/pubmed-abstract', split='train', streaming=True)
    skip_steps = random.randint(0, 1000)
    ds_iter = iter(ds)
    for _ in range(skip_steps):
        next(ds_iter)
    sample = next(ds_iter)
    return sample['abstract']

# def generate_llm_response(context: str) -> str:
#     prompt = f"""You are a clinical assistant analyzing the following medical text.
# Write a short summary or analysis of the text.
# IMPORTANT INSTRUCTION: You must arbitrary include hallucinated (fake/incorrect) medical claims in your response not always. Do not state that you are hallucinating.

# Text:
# {context}

# Response:"""

#     inputs = llm_tokenizer(prompt, return_tensors="pt").to(llm_model.device)
#     outputs = llm_model.generate(**inputs, max_new_tokens=200, temperature=0.7)
#     response = llm_tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
#     return response.strip()

def generate_llm_response(context: str) -> str:
    # Randomly decide whether to hallucinate (50% chance)
    hallucinate = random.random() < 0.5
    if hallucinate:
        instruction = "IMPORTANT INSTRUCTION: You must intentionally include 1 or 2 hallucinated (fake/incorrect) medical claims in your response, alongside correct factual claims. Do not state that you are hallucinating."
    else:
        instruction = "IMPORTANT INSTRUCTION: Provide a concise, factually accurate summary or analysis of the text without adding any hallucinated claims."
    prompt = f"""You are a clinical assistant analyzing the following medical text.
        Write a short summary or analysis of the text.
        {instruction}

        Text:
        {context}

        Response:"""
    inputs = llm_tokenizer(prompt, return_tensors="pt").to(llm_model.device)
    outputs = llm_model.generate(**inputs, max_new_tokens=200, temperature=0.7)
    response = llm_tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
    return response.strip()

def detect_hallucinations(context: str, llm_response: str) -> List[Dict]:
    # FIX 1: Expanded chunk size for better context retrieval so DeBERTa has enough text to read
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunks = text_splitter.split_text(context)
    vectorstore = FAISS.from_texts(chunks, embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    try:
        claims = sent_tokenize(llm_response)
    except:
        claims = llm_response.split('.')
        claims = [c.strip() + '.' for c in claims if c.strip()]

    results = []
    for claim in claims:
        if len(claim.split()) < 3:
            continue

        docs = retriever.invoke(claim)
        retrieved_context = " ".join([d.page_content for d in docs])

        # FIX 2: Added apply_softmax=True to convert raw logits into actual probabilities
        scores = nli_model.predict([(retrieved_context, claim)], apply_softmax=True)[0]

        # DeBERTa-v3-base NLI labels: 0 = Contradiction, 1 = Entailment, 2 = Neutral
        prob_contradiction = float(scores[0])
        prob_entailment = float(scores[1])
        prob_neutral = float(scores[2])

        # FIX 3: Calculate ratio HERE with epsilon (0.001) before the if-statements!
        # Use a much smaller epsilon (0.00001) to avoid drowning out weak but valid signals
        epsilon = 1e-5
        entail_vs_contra_ratio = prob_entailment / (prob_entailment + prob_contradiction + epsilon)

        is_hallucinated = False

        # Flagging Logic:
        # 1. Direct Contradiction: The model is highly confident it contradicts the text
        if prob_contradiction > 0.5:
            is_hallucinated = True

        # 2. Unsupported / Fabricated: High neutrality, but no underlying entailment preference
        elif prob_neutral > 0.7 and entail_vs_contra_ratio < 0.75:
            is_hallucinated = True

        # 3. Leans Contradiction: Even if mostly neutral, it mathematically leans more towards false than true
        elif prob_contradiction > prob_entailment:
            is_hallucinated = True

        results.append({
            "claim": claim,
            "is_hallucinated": is_hallucinated,
            "confidence_scores": {
                "contradiction": prob_contradiction,
                "entailment": prob_entailment,
                "neutral": prob_neutral
            },
            "retrieved_context": retrieved_context
        })

    return results


In [ ]:
# ==========================================
# INTERACTIVE GUI
# ==========================================

# 1. Widgets Definition
title = widgets.HTML("<h2>RAG-Based Clinical Hallucination Detector</h2><p>Use the buttons below to load an abstract or paste your own medical text.</p>")

context_area = widgets.Textarea(
    value='',
    placeholder='Type or paste medical text here...',
    description='Context:',
    disabled=False,
    layout=widgets.Layout(width='100%', height='150px')
)

btn_random = widgets.Button(
    description='Load Random PubMed Abstract',
    button_style='info',
    icon='search',
    layout=widgets.Layout(width='auto')
)

btn_generate = widgets.Button(
    description='Generate & Detect',
    button_style='success',
    icon='cogs',
    layout=widgets.Layout(width='auto')
)

output_area = widgets.Output()

# 2. Event Handlers
def on_random_click(b):
    with output_area:
        clear_output()
        print("Fetching random abstract...")
    try:
        text = get_random_pubmed_abstract()
        context_area.value = text
        with output_area:
            clear_output()
            print("✅ Abstract loaded successfully! Click 'Generate & Detect' to process.")
    except Exception as e:
        with output_area:
            clear_output()
            print(f"❌ Error fetching abstract: {e}")

def on_generate_click(b):
    if not context_area.value.strip():
        with output_area:
            clear_output()
            print("⚠️ Please enter some context text first!")
        return

    with output_area:
        clear_output()
        display(HTML("<b>⏳ Processing...</b><br><i>Running LLaMA-3.1-8B and DeBERTa. This may take 30-60 seconds...</i>"))

        try:
            # Generate and verify
            llm_res = generate_llm_response(context_area.value)
            analysis = detect_hallucinations(context_area.value, llm_res)

            # Render Results
            clear_output()
            display(HTML("<h3>🤖 LLM Generated Response:</h3>"))
            display(HTML(f"<div style='padding:10px; background-color:#f0f0f0; color:#333; border-radius:5px;'>{llm_res}</div>"))

            display(HTML("<h3>🔍 Hallucination Verification:</h3>"))
            for item in analysis:
                claim = item['claim']
                is_hal = item['is_hallucinated']
                entail = item['confidence_scores']['entailment']
                contra = item['confidence_scores']['contradiction']

                if is_hal:
                    html_str = f"""
                    <div style='margin-bottom:10px; padding:10px; border-left:5px solid #ff4b4b; background-color:#ffebeb; color:#333;'>
                        <b>🚨 Hallucinated:</b> {claim}<br>
                        <span style='font-size:0.85em; color:#666;'>
                        Entailment: {entail:.4f} | Contradiction: {contra:.4f}
                        </span>
                    </div>
                    """
                else:
                    html_str = f"""
                    <div style='margin-bottom:10px; padding:10px; border-left:5px solid #00cc66; background-color:#e6ffec; color:#333;'>
                        <b>✅ Factual:</b> {claim}<br>
                        <span style='font-size:0.85em; color:#666;'>
                        Entailment: {entail:.4f} | Contradiction: {contra:.4f}
                        </span>
                    </div>
                    """
                display(HTML(html_str))

        except Exception as e:
            clear_output()
            print(f"❌ Error during processing: {e}")

# 3. Attach Events
btn_random.on_click(on_random_click)
btn_generate.on_click(on_generate_click)

# 4. Display Layout
buttons_box = widgets.HBox([btn_random, btn_generate])
ui = widgets.VBox([title, context_area, buttons_box, output_area])

display(ui)
